In [54]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [55]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [56]:
import pandas as pd
import numpy as np

In [57]:
# custom
from utils import *

# LOAD LETTERS

In [58]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [59]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [60]:
word_df['word'] = word_df['word'].astype(str)

In [61]:
word_df.head()

,word
0,a
1,aa
2,aaa
3,aah
4,aahed


In [62]:
word_df.shape

(370105, 1)

In [63]:
word_df['word'].isna().value_counts()

word
False    370105
Name: count, dtype: int64

In [64]:
word_df['lcase'] = word_df['word'].str.lower()

In [65]:
word_df['n_letters'] = word_df['word'].str.len()

In [66]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [67]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))

In [68]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [69]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [70]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [71]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars
0,abdom,abdom,5,abdmo,"{b, o, m, a, d}",5
1,abend,abend,5,abden,"{b, a, d, n, e}",5
2,abets,abets,5,abest,"{b, t, a, s, e}",5
3,abhor,abhor,5,abhor,"{b, h, o, a, r}",5
4,abide,abide,5,abdei,"{b, i, a, d, e}",5


In [72]:
word_df.shape

(5977, 6)

In [73]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


In [74]:
word_df['word_id'] = range(0, word_df.shape[0])

In [75]:
word_df.shape

(5977, 7)

In [76]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id
0,abdom,abdom,5,abdmo,"{b, o, m, a, d}",5,0
1,abend,abend,5,abden,"{b, a, d, n, e}",5,1
2,abets,abets,5,abest,"{b, t, a, s, e}",5,2
3,abhor,abhor,5,abhor,"{b, h, o, a, r}",5,3
4,abide,abide,5,abdei,"{b, i, a, d, e}",5,4


In [77]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

# BYTE ENCODE WORDS

In [78]:
word_df['word_byte'] = word_df['word'].map(byte_encode_words)

In [79]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id,word_byte
0,abdom,abdom,5,abdmo,"{b, o, m, a, d}",5,0,20491
1,abend,abend,5,abden,"{b, a, d, n, e}",5,1,8219
2,abets,abets,5,abest,"{b, t, a, s, e}",5,2,786451
3,abhor,abhor,5,abhor,"{b, h, o, a, r}",5,3,147587
4,abide,abide,5,abdei,"{b, i, a, d, e}",5,4,283


In [80]:
word_byte_list = word_df['word_byte'].tolist()

## EXAMPLES OF BYTE COMPARISONS

In [81]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [82]:
# no letters in common
w1b & w2b

0

In [83]:
# letters in common
w1b & w3b

147456

In [84]:
w1b | w2b

673975

In [85]:
# this is the same as directly above
testo = byte_encode_words('abhorcleft')
testo

673975

In [86]:
lc_be = byte_encode_words(ascii_lowercase)

In [87]:
lc_be

67108863

In [88]:
word_byte_array = np.array(word_byte_list, dtype = np.int32)

In [89]:
word_byte_to_word_dict = {wb:lcase for wb, lcase in zip(word_df['word_byte'], word_df['lcase'])}

# BUILD LEVEL 2

In [ ]:

l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [ ]:
l2_df['l2'].unique().shape

(640023,)

# BUILD LEVELS 3 THROUGH 5

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
start_pos = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[start_pos, :] = temp_list                   
                    start_pos += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, start_pos)
    


0
0
10000
0
20000
0
30000
0
40000
0
50000
0
60000
0
70000
0
80000
0
90000
48
100000
210
110000
210
120000
228
130000
234
140000
234
150000
486
160000
798
170000
1026
180000
1152
190000
1458
200000
1740
210000
2274
220000
2364
230000
2544
240000
2616
250000
3786
260000
5508
270000
6174
280000
6342
290000
6570
300000
7074
310000
8442
320000
9024
330000
9024
340000
9042
350000
9114
360000
9552
370000
9552
380000
9552
390000
9576
400000
9576
410000
9600
420000
9636
430000
9678
440000
9972
450000
10200
460000
10200
470000
11472
480000
18966
490000
19044
500000
19272
510000
20004
520000
20352
530000
20802
540000
21228
550000
21642
560000
21930
570000
22674
580000
23016
590000
23292
600000
23430
610000
23814
620000
24450
630000
25272
640000
27096


# CREATE AND SAVE OUTPUT

In [ ]:
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [109]:
# save all dataframes to disk
l2_df.to_csv(path_or_buf='l2.txt', sep = '\t', index = False)
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)


In [110]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327


In [111]:
# how many unique letter groups?
for il in range(2, 6):
    cn  = f"l{il}"
    print(cn, l5_df[cn].unique().shape)

l2 (2006,)
l3 (2132,)
l4 (650,)
l5 (12,)


In [112]:
# get words

In [113]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [99]:
# get unique word combos
output_list = []
for i_row, row in l5_df.iterrows():
    my_set = set()
    for cn_idx in range(1, 6):
        cn = f"w{cn_idx}"
        my_set.add(row[cn])

    output = tuple(sorted(my_set))

    output_list.append(output)

    

In [100]:
l5_df['word_group'] = output_list

In [101]:
l5_df = l5_df.sort_values(by = ['w1', 'w2', 'w3', 'w4', 'w5']).reset_index(drop = True)

In [102]:
l5_df['word_group_hash'] = l5_df['word_group'].map(hash)

In [103]:
l5_df['word_group_hash'].unique().shape

(539,)

In [104]:
l5_df = l5_df.drop_duplicates(subset = 'word_group_hash').reset_index(drop = True)

In [105]:
l5_df.to_excel(excel_writer='l5_output.xlsx', index = False)

# EXPAND THE INPUTS

In [119]:
l2_df.shape

(3213696, 3)

In [120]:
test_l2_df.shape

(640023, 3)

In [121]:
test_1_l2_df = l2_df.loc[l2_df['l2'].isin(l5_df['l2']), :]

In [122]:
test_1_l2_df.shape

(2814, 3)

In [123]:
test_2_l2_df = test_l2_df.loc[test_l2_df['l2'].isin(l5_df['l2']), :]

In [124]:
test_2_l2_df.shape

(2005, 3)

# build a cool graph!

In [ ]:
# update l2_df with the words

In [133]:
for ii in range(1,3):
    cn = f"w{ii}b"
    ncn = f"w{ii}"
    print(cn)
    l2_df[ncn] = l2_df[cn].map(word_byte_to_word_dict)
    test_1_l2_df[ncn] = test_1_l2_df[cn].map(word_byte_to_word_dict)

w1b
w2b


C:\Users\babbm\AppData\Local\Temp\ipykernel_4280\1166389756.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_1_l2_df[ncn] = test_1_l2_df[cn].map(word_byte_to_word_dict)
C:\Users\babbm\AppData\Local\Temp\ipykernel_4280\1166389756.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_1_l2_df[ncn] = test_1_l2_df[cn].map(word_byte_to_word_dict)


In [134]:
l2_df.shape

(3213696, 5)

In [135]:
l2_df.head()

,w1b,w2b,l2,w1,w2
0,20491,264468,284959,abdom,ceils
1,20491,532756,553247,abdom,ceint
2,20491,788500,808991,abdom,celts
3,20491,794644,815135,abdom,cents
4,20491,1114388,1134879,abdom,cequi


In [136]:
# build a tuple
test_1_l2_df    

,w1b,w2b,l2,w1,w2
193756,16912387,8914984,25827371,ambry,fldxt
194339,16912387,1344516,18256903,ambry,pucks
194350,16912387,1351744,18264131,ambry,pungs
194567,16912387,35668496,52580883,ambry,vejoz
194612,16912387,4195716,21108103,ambry,whick
...,...,...,...,...,...
3212561,4326420,33562945,37889365,wreck,zigan
3212563,4326420,50340160,54666580,wreck,zingy
3212637,4326660,50356288,54682948,wrick,zygon
3212980,23199760,33562945,56762705,wyver,zigan


In [137]:
test_1_l2_df['l2'].unique().shape

(2005,)

In [138]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327,ambry,fldxt,pucks,whing,vejoz
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327,ambry,fldxt,pungs,whick,vejoz
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327,ambry,fldxt,vejoz,pucks,whing


In [139]:
# build the groups

In [140]:
def build_a_word_tuple(row, level:int):
    cn_list = []
    for ii in range(1, level + 1):
        cn = f"w{ii}"
        cn_list.append(row[cn])

    cn_list = sorted(cn_list)
    return tuple(cn_list)


In [144]:
l5_df['l2_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 2)
l5_df['l3_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 3)
l5_df['l4_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 4)
l5_df['l5_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 5)

In [145]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,l2_words,l3_words,l4_words,l5_words
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt)","(ambry, fldxt, pucks)","(ambry, fldxt, pucks, vejoz)","(ambry, fldxt, pucks, vejoz, whing)"
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327,ambry,fldxt,pucks,whing,vejoz,"(ambry, fldxt)","(ambry, fldxt, pucks)","(ambry, fldxt, pucks, whing)","(ambry, fldxt, pucks, vejoz, whing)"
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt)","(ambry, fldxt, pungs)","(ambry, fldxt, pungs, vejoz)","(ambry, fldxt, pungs, vejoz, whick)"
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327,ambry,fldxt,pungs,whick,vejoz,"(ambry, fldxt)","(ambry, fldxt, pungs)","(ambry, fldxt, pungs, whick)","(ambry, fldxt, pungs, vejoz, whick)"
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327,ambry,fldxt,vejoz,pucks,whing,"(ambry, fldxt)","(ambry, fldxt, vejoz)","(ambry, fldxt, pucks, vejoz)","(ambry, fldxt, pucks, vejoz, whing)"


In [146]:
import networkx as nx

In [147]:
my_graph = nx.DiGraph()

In [152]:
dir(l5_df)

['T',
 '_AXIS_LEN',
 '_AXIS_ORDERS',
 '_AXIS_TO_AXIS_NUMBER',
 '_HANDLED_TYPES',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_ufunc__',
 '__arrow_c_stream__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__dataframe__',
 '__dataframe_consortium_standard__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__finalize__',
 '__firstlineno__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__imod__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pandas_priority__',
 '__pos__

In [ ]:
testo = l5_df[['l2_words', 'l3_words']].to

In [156]:
testo

rec.array([(     0, ('ambry', 'fldxt'), ('ambry', 'fldxt', 'pucks')),
           (     1, ('ambry', 'fldxt'), ('ambry', 'fldxt', 'pucks')),
           (     2, ('ambry', 'fldxt'), ('ambry', 'fldxt', 'pungs')), ...,
           (999997, (nan, nan), (nan, nan, nan)),
           (999998, (nan, nan), (nan, nan, nan)),
           (999999, (nan, nan), (nan, nan, nan))],
          dtype=[('index', '<i8'), ('l2_words', 'O'), ('l3_words', 'O')])

In [ ]:
my_graph.add_edges_from(].to_records())

ValueError: dictionary update sequence element #0 has length 5; 2 is required